
# Notebook 09 — Tableau Dashboard Reporting

## Objective
Transform the analytical outputs from Notebooks 01–08
into Tableau-ready reporting datasets.

## Dashboard Architecture
1. Network Cost & Optimization Performance
2. Sourcing & Manufacturing Intelligence
3. Transportation & Distribution Intelligence
4. Risk, Scenario & Sensitivity Intelligence

## Data Preparation Methodology
- Load validated network and optimization outputs
- Standardize business dimensions and metric names
- Define one consistent row grain per reporting table
- Reconcile financial and operational KPIs
- Preserve optimal versus non-optimal scenario statuses
- Export independent, Tableau-ready CSV datasets
- Generate a data dictionary and dashboard mapping

## Reporting Rules
- No fabricated or simulated results are added here
- Financial metrics come from solved optimization models
- All cost results reflect hypothetical network assumptions
- Scenario costs are not probabilities or expected losses
- Demand-growth costs include additional fulfilled units
- Avoid summing repeated KPIs across unrelated fact tables

## Deliverables
16 reporting CSVs, a reporting data dictionary,
an executive summary, and validation checks.


In [1]:
# STEP 2: Initialize Tableau reporting pipeline

import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display, Markdown

# --------------------------------------------------
# 1. Locate Project 2 repository
# --------------------------------------------------

current_path = Path.cwd().resolve()

repo_root = next(
    (
        p for p in [current_path, *current_path.parents]
        if (p / "data/assumptions/bom.csv").is_file()
    ),
    None
)

assert repo_root is not None, "Project 2 repository not found"

# --------------------------------------------------
# 2. Define input datasets
# --------------------------------------------------

source_files = {
    # Network and financial performance
    "cost_bridge":
        "outputs/tables/transport_cost_savings_bridge.csv",
    "distribution_kpis":
        "outputs/tables/distribution_performance_kpis.csv",

    # Supplier and manufacturing
    "suppliers":
        "data/assumptions/suppliers.csv",
    "plants":
        "data/assumptions/plants.csv",
    "reference_orders":
        "outputs/tables/scenario_purchase_orders.csv",
    "reference_production":
        "outputs/tables/scenario_production.csv",
    "reference_component_inventory":
        "outputs/tables/scenario_component_inventory.csv",

    # Transportation and distribution
    "freight_modes":
        "outputs/tables/transport_mode_comparison.csv",
    "freight_lanes":
        "outputs/tables/transport_lane_comparison.csv",
    "transport_weekly":
        "outputs/tables/transport_weekly_timeline.csv",
    "dc_weekly":
        "outputs/tables/distribution_inventory_comparison.csv",

    # Scenario analysis
    "scenario_summary":
        "outputs/tables/scenario_comparison.csv",
    "scenario_audit":
        "outputs/tables/scenario_audit.csv",
    "scenario_operations":
        "outputs/tables/scenario_operational_response.csv",
    "scenario_costs":
        "outputs/tables/scenario_cost_breakdown.csv",

    # Sensitivity analysis
    "sensitivity_summary":
        "outputs/tables/sensitivity_comparison.csv",
    "sensitivity_costs":
        "outputs/tables/sensitivity_cost_breakdown.csv",
    "capacity_value":
        "outputs/tables/sensitivity_capacity_investment_value.csv",
    "freight_sensitivity":
        "outputs/tables/sensitivity_freight_economics.csv",
    "demand_sensitivity":
        "outputs/tables/sensitivity_demand_growth.csv"
}

# --------------------------------------------------
# 3. Verify and load all source files
# --------------------------------------------------

missing = [
    relative_path
    for relative_path in source_files.values()
    if not (repo_root / relative_path).is_file()
]

assert not missing, (
    "Missing source files:\n"
    + "\n".join(missing)
    + "\nRun the corresponding earlier notebook."
)

sources = {
    name: pd.read_csv(repo_root / relative_path)
    for name, relative_path in source_files.items()
}

assert all(len(df) > 0 for df in sources.values())

# --------------------------------------------------
# 4. Create Tableau output directories
# --------------------------------------------------

tableau_root = repo_root / "outputs" / "tableau"

dashboard_dirs = {
    "01_cost": tableau_root / "01_cost",
    "02_sourcing": tableau_root / "02_sourcing",
    "03_distribution": tableau_root / "03_distribution",
    "04_risk": tableau_root / "04_risk"
}

for folder in dashboard_dirs.values():
    folder.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 5. Reference financial values
# --------------------------------------------------

cost_bridge = sources["cost_bridge"].copy()

baseline_cost = float(
    cost_bridge["Baseline_Cost_USD"].sum()
)

optimized_cost = float(
    cost_bridge["Optimized_Cost_USD"].sum()
)

total_savings = baseline_cost - optimized_cost

assert np.isclose(baseline_cost, 414139.70, atol=0.01)
assert np.isclose(optimized_cost, 404131.20, atol=0.01)
assert np.isclose(total_savings, 10008.50, atol=0.01)

# --------------------------------------------------
# 6. Display initialization
# --------------------------------------------------

print("NOTEBOOK 09 INITIALIZED SUCCESSFULLY")
print("=" * 50)
print("Source datasets loaded:", len(sources))
print("Tableau dashboard folders:", len(dashboard_dirs))
print(f"Baseline cost: ${baseline_cost:,.2f}")
print(f"Optimized cost: ${optimized_cost:,.2f}")
print(f"Total savings: ${total_savings:,.2f}")
print("STEP 2: PASSED")

display(pd.DataFrame([
    {"Dataset": name, "Rows": len(df)}
    for name, df in sources.items()
]))

NOTEBOOK 09 INITIALIZED SUCCESSFULLY
Source datasets loaded: 20
Tableau dashboard folders: 4
Baseline cost: $414,139.70
Optimized cost: $404,131.20
Total savings: $10,008.50
STEP 2: PASSED


,Dataset,Rows
0,cost_bridge,4
1,distribution_kpis,2
2,suppliers,6
3,plants,1
4,reference_orders,360
5,reference_production,60
6,reference_component_inventory,180
7,freight_modes,8
8,freight_lanes,4
9,transport_weekly,48


In [2]:
# STEP 3A: Network cost and executive KPI datasets

def require_columns(df, columns, label):
    missing = sorted(set(columns) - set(df.columns))
    assert not missing, f"{label} missing columns: {missing}"


def check_grain(df, keys, label):
    require_columns(df, keys, label)
    assert df[keys].notna().all().all(), (
        f"{label} has missing keys"
    )
    assert not df.duplicated(keys).any(), (
        f"{label} contains duplicate rows for {keys}"
    )


require_columns(
    cost_bridge,
    [
        "Cost_Category",
        "Baseline_Cost_USD",
        "Optimized_Cost_USD",
        "Savings_USD"
    ],
    "Cost bridge"
)

check_grain(cost_bridge, ["Cost_Category"], "Cost bridge")

# Table 1: one row per cost category and plan
network_cost_categories = cost_bridge.melt(
    id_vars="Cost_Category",
    value_vars=[
        "Baseline_Cost_USD",
        "Optimized_Cost_USD"
    ],
    var_name="Plan",
    value_name="Cost_USD"
)

network_cost_categories["Plan"] = (
    network_cost_categories["Plan"].replace({
        "Baseline_Cost_USD": "Baseline",
        "Optimized_Cost_USD": "Optimized"
    })
)

category_order = {
    "Procurement": 1,
    "Manufacturing": 2,
    "Transportation": 3,
    "DC Inventory Holding": 4
}

network_cost_categories["Category_Order"] = (
    network_cost_categories["Cost_Category"]
    .map(category_order)
)

assert network_cost_categories[
    "Category_Order"
].notna().all()

check_grain(
    network_cost_categories,
    ["Plan", "Cost_Category"],
    "Network cost categories"
)

# Table 2: one row per plan
network_kpis = (
    network_cost_categories.groupby(
        "Plan", as_index=False
    )["Cost_USD"]
    .sum()
    .rename(columns={
        "Cost_USD": "Total_Network_Cost_USD"
    })
)

distribution_metrics = sources[
    "distribution_kpis"
].copy()

require_columns(
    distribution_metrics,
    [
        "Plan",
        "Customer_Demand_Units",
        "Fulfilled_Units",
        "Service_Level_Pct",
        "Shipped_Units",
        "Expedited_Share_Pct",
        "Freight_Cost_USD"
    ],
    "Distribution KPIs"
)

network_kpis = network_kpis.merge(
    distribution_metrics,
    on="Plan",
    how="left",
    validate="one_to_one"
)

network_kpis["Baseline_Reference_Cost_USD"] = (
    baseline_cost
)

network_kpis["Savings_vs_Baseline_USD"] = np.where(
    network_kpis["Plan"] == "Optimized",
    baseline_cost - network_kpis["Total_Network_Cost_USD"],
    0
)

network_kpis["Cost_Improvement_Pct"] = (
    network_kpis["Savings_vs_Baseline_USD"]
    / baseline_cost * 100
)

network_kpis["Plan_Order"] = (
    network_kpis["Plan"]
    .map({"Baseline": 1, "Optimized": 2})
)

network_kpis = network_kpis.sort_values(
    "Plan_Order"
).reset_index(drop=True)

# Table 3: one row per savings category
network_savings_bridge = cost_bridge.copy()

network_savings_bridge["Category_Order"] = (
    network_savings_bridge["Cost_Category"]
    .map(category_order)
)

network_savings_bridge["Savings_Share_of_Net_Pct"] = (
    network_savings_bridge["Savings_USD"]
    / total_savings * 100
)

# Validate all totals
assert len(network_kpis) == 2

assert np.isclose(
    network_cost_categories[
        "Cost_USD"
    ].sum(),
    baseline_cost + optimized_cost
)

assert np.isclose(
    network_savings_bridge[
        "Savings_USD"
    ].sum(),
    total_savings
)

assert network_kpis[
    "Service_Level_Pct"
].eq(100).all()

print("DASHBOARD 1 DATASETS CREATED")
print("=" * 45)

display(network_kpis.round(2))
display(network_savings_bridge.round(2))

print("\nSTEP 3A: PASSED")


DASHBOARD 1 DATASETS CREATED


,Plan,Total_Network_Cost_USD,Customer_Demand_Units,Fulfilled_Units,Service_Level_Pct,Shipped_Units,Expedited_Units,Expedited_Share_Pct,Avg_Ending_DC_Inventory,Min_Storage_Headroom,DC_Holding_Cost_USD,Freight_Cost_USD,Baseline_Reference_Cost_USD,Savings_vs_Baseline_USD,Cost_Improvement_Pct,Plan_Order
0,Baseline,414139.7,6180,6180,100.0,5665,2266,40.00,45.0,120,718.2,34556.5,414139.7,0.0,0.00,1
1,Optimized,404131.2,6180,6180,100.0,5665,562,9.92,7.5,120,119.7,25656.5,414139.7,10008.5,2.42,2


,Cost_Category,Baseline_Cost_USD,Optimized_Cost_USD,Savings_USD,Category_Order,Savings_Share_of_Net_Pct
0,Procurement,254235.0,253725.0,510.0,1,5.10
1,Manufacturing,124630.0,124630.0,0.0,2,0.00
2,Transportation,34556.5,25656.5,8900.0,3,88.92
3,DC Inventory Holding,718.2,119.7,598.5,4,5.98



STEP 3A: PASSED


In [3]:
# STEP 4A: Sourcing and manufacturing reporting

# --------------------------------------------------
# 1. Supplier weekly purchase decisions
# --------------------------------------------------

supplier_weekly = sources[
    "reference_orders"
].loc[
    lambda df: df["Scenario"] == "Reference"
].copy()

require_columns(
    supplier_weekly,
    [
        "Supplier_ID", "Component_ID", "Order_Week",
        "Arrival_Week", "Order_Units",
        "Purchase_Cost_USD"
    ],
    "Reference supplier orders"
)

supplier_master = sources["suppliers"][[
    "Supplier_ID",
    "Component_ID",
    "Weekly_Capacity_Units",
    "Lead_Time_Weeks",
    "Unit_Cost_USD"
]].copy()

supplier_weekly = supplier_weekly.merge(
    supplier_master,
    on=["Supplier_ID", "Component_ID"],
    how="left",
    validate="many_to_one"
)

assert supplier_weekly.notna().all().all()

supplier_weekly["Capacity_Utilization_Pct"] = (
    supplier_weekly["Order_Units"]
    / supplier_weekly["Weekly_Capacity_Units"] * 100
)

supplier_weekly["Week"] = (
    supplier_weekly["Order_Week"]
)

supplier_weekly["Week_Label"] = (
    "W" + supplier_weekly["Week"]
    .astype(int).astype(str).str.zfill(2)
)

assert (
    supplier_weekly["Capacity_Utilization_Pct"]
    <= 100.00001
).all()

check_grain(
    supplier_weekly,
    ["Supplier_ID", "Component_ID", "Order_Week"],
    "Supplier weekly orders"
)

# --------------------------------------------------
# 2. Manufacturing weekly decisions
# --------------------------------------------------

manufacturing_weekly = sources[
    "reference_production"
].loc[
    lambda df: df["Scenario"] == "Reference"
].copy()

assembly_cost = float(
    sources["plants"][
        "Assembly_Cost_Per_Unit_USD"
    ].iloc[0]
)

manufacturing_weekly["Production_Cost_USD"] = (
    manufacturing_weekly["Production_Units"]
    * assembly_cost
)

manufacturing_weekly["Capacity_Utilization_Pct"] = (
    manufacturing_weekly["Production_Units"]
    / manufacturing_weekly["Available_Capacity"] * 100
)

manufacturing_weekly["Unused_Capacity_Units"] = (
    manufacturing_weekly["Available_Capacity"]
    - manufacturing_weekly["Production_Units"]
)

manufacturing_weekly["Week_Label"] = (
    "W" + manufacturing_weekly["Week"]
    .astype(int).astype(str).str.zfill(2)
)

check_grain(
    manufacturing_weekly,
    ["Week"],
    "Manufacturing weekly"
)

assert len(manufacturing_weekly) == 12
assert (
    manufacturing_weekly["Unused_Capacity_Units"] >= 0
).all()

# --------------------------------------------------
# 3. Component inventory reporting
# --------------------------------------------------

component_weekly = sources[
    "reference_component_inventory"
].loc[
    lambda df: df["Scenario"] == "Reference"
].copy()

component_weekly["Week_Label"] = (
    "W" + component_weekly["Week"]
    .astype(int).astype(str).str.zfill(2)
)

component_weekly["Stockout_Buffer_Flag"] = (
    component_weekly["Ending_Inventory"] == 0
).astype(int)

check_grain(
    component_weekly,
    ["Component_ID", "Week"],
    "Component inventory weekly"
)

assert len(component_weekly) == 36
assert (
    component_weekly["Ending_Inventory"] >= 0
).all()

# --------------------------------------------------
# 4. Financial reconciliations
# --------------------------------------------------

reference_cost_categories = sources[
    "scenario_costs"
].loc[
    lambda df: df["Scenario"] == "Reference"
].set_index("Cost_Category")["Cost_USD"]

assert np.isclose(
    supplier_weekly["Purchase_Cost_USD"].sum(),
    reference_cost_categories["Procurement"],
    atol=0.01
)

assert np.isclose(
    manufacturing_weekly["Production_Cost_USD"].sum(),
    reference_cost_categories["Manufacturing"],
    atol=0.01
)

print("DASHBOARD 2 DATASETS CREATED")
print("=" * 45)
print("Supplier records:", len(supplier_weekly))
print("Manufacturing weeks:", len(manufacturing_weekly))
print("Component inventory records:", len(component_weekly))
print("Procurement reconciliation: PASSED")
print("Manufacturing reconciliation: PASSED")

display(
    supplier_weekly.head(8).round(2)
)

print("\nSTEP 4A: PASSED")


DASHBOARD 2 DATASETS CREATED
Supplier records: 72
Manufacturing weeks: 12
Component inventory records: 36
Procurement reconciliation: PASSED
Manufacturing reconciliation: PASSED


,Scenario,Supplier_ID,Component_ID,Order_Week,Arrival_Week,Order_Units,Unit_Cost_USD_x,Purchase_Cost_USD,Weekly_Capacity_Units,Lead_Time_Weeks,Unit_Cost_USD_y,Capacity_Utilization_Pct,Week,Week_Label
0,Reference,SUP_A,ELEC_001,1,3,450,32.0,14400.0,450,2,32,100.0,1,W01
1,Reference,SUP_A,ELEC_001,2,4,450,32.0,14400.0,450,2,32,100.0,2,W02
2,Reference,SUP_A,ELEC_001,3,5,450,32.0,14400.0,450,2,32,100.0,3,W03
3,Reference,SUP_A,ELEC_001,4,6,450,32.0,14400.0,450,2,32,100.0,4,W04
4,Reference,SUP_A,ELEC_001,5,7,450,32.0,14400.0,450,2,32,100.0,5,W05
5,Reference,SUP_A,ELEC_001,6,8,450,32.0,14400.0,450,2,32,100.0,6,W06
6,Reference,SUP_A,ELEC_001,7,9,450,32.0,14400.0,450,2,32,100.0,7,W07
7,Reference,SUP_A,ELEC_001,8,10,450,32.0,14400.0,450,2,32,100.0,8,W08



STEP 4A: PASSED


In [4]:
# STEP 5A: Transportation and distribution reporting

# --------------------------------------------------
# 1. Freight mode comparison
# --------------------------------------------------

freight_modes = sources["freight_modes"].copy()

check_grain(
    freight_modes,
    ["Plan", "DC_ID", "Transport_Mode"],
    "Freight mode comparison"
)

# --------------------------------------------------
# 2. Freight lane comparison
# --------------------------------------------------

freight_lanes = sources["freight_lanes"].copy()

check_grain(
    freight_lanes,
    ["Lane_ID"],
    "Freight lane comparison"
)

# --------------------------------------------------
# 3. Weekly distribution operations
# --------------------------------------------------

dc_weekly = sources["dc_weekly"].copy()

transport_weekly = sources[
    "transport_weekly"
][[
    "Plan",
    "DC_ID",
    "Week",
    "Departures_Units",
    "Departures_Freight_USD"
]].copy()

dc_weekly = dc_weekly.merge(
    transport_weekly,
    on=["Plan", "DC_ID", "Week"],
    how="left",
    validate="one_to_one"
)

require_columns(
    dc_weekly,
    [
        "Plan", "DC_ID", "Week",
        "Customer_Demand", "Fulfilled_Units",
        "Shipment_Arrivals", "Arrivals_Units",
        "Beginning_Inventory", "Ending_Inventory",
        "Pre_Demand_Inventory",
        "Storage_Capacity_Units",
        "Storage_Headroom_Units",
        "Departures_Units",
        "Departures_Freight_USD"
    ],
    "DC weekly reporting"
)

assert dc_weekly.notna().all().all()

dc_weekly["Week_Label"] = (
    "W" + dc_weekly["Week"]
    .astype(int).astype(str).str.zfill(2)
)

# Utilization before customer demand is fulfilled
dc_weekly["Storage_Utilization_Pct"] = (
    dc_weekly["Pre_Demand_Inventory"]
    / dc_weekly["Storage_Capacity_Units"] * 100
)

dc_weekly["Service_Level_Pct"] = (
    dc_weekly["Fulfilled_Units"]
    / dc_weekly["Customer_Demand"] * 100
)

# Reconcile arrival records
assert np.allclose(
    dc_weekly["Shipment_Arrivals"],
    dc_weekly["Arrivals_Units"]
)

# Check customer service and storage
assert np.allclose(
    dc_weekly["Service_Level_Pct"], 100
)

assert (
    dc_weekly["Storage_Headroom_Units"] >= 0
).all()

assert (
    dc_weekly["Storage_Utilization_Pct"] <= 100.00001
).all()

check_grain(
    dc_weekly,
    ["Plan", "DC_ID", "Week"],
    "Weekly DC operations"
)

assert len(dc_weekly) == 48

# --------------------------------------------------
# 4. Freight cost reconciliation
# --------------------------------------------------

freight_totals = (
    freight_modes.groupby("Plan")[
        "Freight_Cost_USD"
    ].sum()
)

weekly_freight_totals = (
    dc_weekly.groupby("Plan")[
        "Departures_Freight_USD"
    ].sum()
)

for plan, expected_cost in [
    (
        "Baseline",
        cost_bridge.loc[
            cost_bridge["Cost_Category"] ==
            "Transportation",
            "Baseline_Cost_USD"
        ].iloc[0]
    ),
    (
        "Optimized",
        cost_bridge.loc[
            cost_bridge["Cost_Category"] ==
            "Transportation",
            "Optimized_Cost_USD"
        ].iloc[0]
    )
]:
    assert np.isclose(
        freight_totals.loc[plan],
        expected_cost,
        atol=0.01
    )

    assert np.isclose(
        weekly_freight_totals.loc[plan],
        expected_cost,
        atol=0.01
    )

print("DASHBOARD 3 DATASETS CREATED")
print("=" * 45)
print("Mode comparison rows:", len(freight_modes))
print("Lane comparison rows:", len(freight_lanes))
print("Weekly DC records:", len(dc_weekly))
print("Freight reconciliation: PASSED")
print("Customer fulfillment: PASSED")
print("Storage capacity: PASSED")

display(dc_weekly.head(8).round(2))

print("\nSTEP 5A: PASSED")


DASHBOARD 3 DATASETS CREATED
Mode comparison rows: 8
Lane comparison rows: 4
Weekly DC records: 48
Freight reconciliation: PASSED
Customer fulfillment: PASSED
Storage capacity: PASSED


,Week,DC_ID,Beginning_Inventory,Shipment_Arrivals,Customer_Demand,Fulfilled_Units,Shortage_Units,Ending_Inventory,Plan,Arrivals_Units,...,Holding_Cost_Per_Unit_Per_Week_USD,Pre_Demand_Inventory,Storage_Headroom_Units,Demand_Units,Holding_Cost_USD,Departures_Units,Departures_Freight_USD,Week_Label,Storage_Utilization_Pct,Service_Level_Pct
0,1,DC_EAST,330,0,309,309,0,21,Baseline,0.0,...,1.25,330,120,309,26.25,309.0,1081.5,W01,73.33,100.0
1,2,DC_EAST,21,309,309,309,0,21,Baseline,309.0,...,1.25,330,120,309,26.25,309.0,1081.5,W02,73.33,100.0
2,3,DC_EAST,21,309,309,309,0,21,Baseline,309.0,...,1.25,330,120,309,26.25,309.0,1081.5,W03,73.33,100.0
3,4,DC_EAST,21,309,309,309,0,21,Baseline,309.0,...,1.25,330,120,309,26.25,309.0,1081.5,W04,73.33,100.0
4,5,DC_EAST,21,309,309,309,0,21,Baseline,309.0,...,1.25,330,120,309,26.25,309.0,1081.5,W05,73.33,100.0
5,6,DC_EAST,21,309,309,309,0,21,Baseline,309.0,...,1.25,330,120,309,26.25,309.0,1081.5,W06,73.33,100.0
6,7,DC_EAST,21,309,309,309,0,21,Baseline,309.0,...,1.25,330,120,309,26.25,309.0,1081.5,W07,73.33,100.0
7,8,DC_EAST,21,309,309,309,0,21,Baseline,309.0,...,1.25,330,120,309,26.25,309.0,1081.5,W08,73.33,100.0



STEP 5A: PASSED


In [5]:
# STEP 6A: Build scenario intelligence datasets

scenario_summary = sources[
    "scenario_summary"
].copy()

scenario_audit = sources[
    "scenario_audit"
].copy()

scenario_operations = sources[
    "scenario_operations"
].copy()

# --------------------------------------------------
# 1. Combine cost, audit and operational metrics
# --------------------------------------------------

scenario_summary = scenario_summary.merge(
    scenario_audit[[
        "Scenario",
        "Solver_Status",
        "Independent_Audit"
    ]],
    on="Scenario",
    how="left",
    validate="one_to_one"
)

scenario_summary = scenario_summary.merge(
    scenario_operations,
    on="Scenario",
    how="left",
    validate="one_to_one"
)

scenario_summary["Is_Optimal"] = (
    scenario_summary["Status"] == "Optimal"
).astype(int)

scenario_summary["Cost_Per_Demand_Unit_USD"] = (
    scenario_summary["Optimized_Cost_USD"]
    / scenario_summary["Total_Demand_Units"]
)

scenario_labels = {
    "Reference": "Original Optimized Network",
    "Supplier_Disruption": "Supplier Order Interruption",
    "Demand_Surge": "Regional Demand Surge",
    "Manufacturing_Disruption": "Plant Capacity Reduction",
    "Freight_Inflation": "Standard Freight Inflation"
}

scenario_summary["Display_Name"] = (
    scenario_summary["Scenario"]
    .replace(scenario_labels)
)

check_grain(
    scenario_summary,
    ["Scenario"],
    "Scenario summary"
)

assert len(scenario_summary) == 5

assert (
    scenario_summary.loc[
        scenario_summary["Is_Optimal"] == 1,
        "Independent_Audit"
    ] == "PASSED"
).all()

# --------------------------------------------------
# 2. Scenario costs by category
# --------------------------------------------------

scenario_costs = sources[
    "scenario_costs"
].copy()

reference_category_costs = scenario_costs.loc[
    scenario_costs["Scenario"] == "Reference",
    ["Cost_Category", "Cost_USD"]
].rename(columns={
    "Cost_USD": "Reference_Category_Cost_USD"
})

scenario_costs = scenario_costs.merge(
    reference_category_costs,
    on="Cost_Category",
    how="left",
    validate="many_to_one"
)

scenario_costs["Incremental_Category_Cost_USD"] = (
    scenario_costs["Cost_USD"]
    - scenario_costs["Reference_Category_Cost_USD"]
)

check_grain(
    scenario_costs,
    ["Scenario", "Cost_Category"],
    "Scenario cost categories"
)

# --------------------------------------------------
# 3. Reconcile scenario costs
# --------------------------------------------------

scenario_totals = (
    scenario_costs.groupby("Scenario")[
        "Cost_USD"
    ].sum()
)

for row in scenario_summary.itertuples(index=False):

    if row.Status == "Optimal":
        assert np.isclose(
            scenario_totals.loc[row.Scenario],
            row.Optimized_Cost_USD,
            atol=0.01
        )

print("SCENARIO INTELLIGENCE DATASETS CREATED")
print("=" * 45)
print("Scenario records:", len(scenario_summary))
print("Scenario cost records:", len(scenario_costs))
print("Scenario cost reconciliation: PASSED")

display(
    scenario_summary[[
        "Scenario",
        "Status",
        "Total_Demand_Units",
        "Optimized_Cost_USD",
        "Incremental_Cost_USD",
        "Independent_Audit"
    ]].round(2)
)

print("\nSTEP 6A: PASSED")


SCENARIO INTELLIGENCE DATASETS CREATED
Scenario records: 5
Scenario cost records: 20
Scenario cost reconciliation: PASSED


,Scenario,Status,Total_Demand_Units,Optimized_Cost_USD,Incremental_Cost_USD,Independent_Audit
0,Reference,Optimal,6180,404131.20,0.00,PASSED
1,Supplier_Disruption,Optimal,6180,417631.20,13500.00,PASSED
2,Demand_Surge,Optimal,6336,418967.70,14836.50,PASSED
3,Manufacturing_Disruption,Optimal,6180,405741.20,1610.00,PASSED
4,Freight_Inflation,Optimal,6180,413920.45,9789.25,PASSED



STEP 6A: PASSED


In [6]:
# STEP 6B: Build sensitivity intelligence datasets

# --------------------------------------------------
# 1. Overall sensitivity experiment summary
# --------------------------------------------------

sensitivity_summary = sources[
    "sensitivity_summary"
].copy()

sensitivity_summary["Is_Optimal"] = (
    sensitivity_summary["Status"] == "Optimal"
).astype(int)

sensitivity_summary["Reference_Cost_USD"] = (
    optimized_cost
)

sensitivity_summary["Savings_vs_Reference_USD"] = (
    optimized_cost
    - sensitivity_summary["Optimized_Cost_USD"]
)

check_grain(
    sensitivity_summary,
    ["Experiment"],
    "Sensitivity summary"
)

assert len(sensitivity_summary) == 18

assert (
    sensitivity_summary.loc[
        sensitivity_summary["Is_Optimal"] == 1,
        "Independent_Audit"
    ] == "PASSED"
).all()

# --------------------------------------------------
# 2. Sensitivity costs by category
# --------------------------------------------------

sensitivity_costs = sources[
    "sensitivity_costs"
].copy()

check_grain(
    sensitivity_costs,
    ["Experiment", "Cost_Category"],
    "Sensitivity cost categories"
)

# --------------------------------------------------
# 3. Capacity investment economics
# --------------------------------------------------

capacity_value = sources[
    "capacity_value"
].copy()

check_grain(
    capacity_value,
    ["Category", "Lever", "Change_Pct"],
    "Capacity investment value"
)

assert len(capacity_value) == 9

# --------------------------------------------------
# 4. Freight economics
# --------------------------------------------------

freight_sensitivity = sources[
    "freight_sensitivity"
].copy()

check_grain(
    freight_sensitivity,
    ["Experiment"],
    "Freight sensitivity"
)

assert len(freight_sensitivity) == 4

# --------------------------------------------------
# 5. Demand growth sensitivity
# --------------------------------------------------

demand_sensitivity = sources[
    "demand_sensitivity"
].copy()

check_grain(
    demand_sensitivity,
    ["Experiment"],
    "Demand sensitivity"
)

assert len(demand_sensitivity) == 4

# --------------------------------------------------
# 6. Reconcile costs for optimal experiments
# --------------------------------------------------

sensitivity_totals = (
    sensitivity_costs.groupby("Experiment")[
        "Cost_USD"
    ].sum()
)

for row in sensitivity_summary.itertuples(index=False):

    if row.Status == "Optimal":

        assert np.isclose(
            sensitivity_totals.loc[row.Experiment],
            row.Optimized_Cost_USD,
            atol=0.01
        )

print("SENSITIVITY INTELLIGENCE DATASETS CREATED")
print("=" * 45)
print("Sensitivity experiments:", len(sensitivity_summary))
print("Capacity valuation rows:", len(capacity_value))
print("Freight experiments:", len(freight_sensitivity))
print("Demand experiments:", len(demand_sensitivity))
print("Sensitivity cost reconciliation: PASSED")

display(
    sensitivity_summary[[
        "Experiment",
        "Category",
        "Change_Pct",
        "Status",
        "Optimized_Cost_USD",
        "Cost_Change_USD"
    ]].round(2)
)

print("\nSTEP 6B: PASSED")

SENSITIVITY INTELLIGENCE DATASETS CREATED
Sensitivity experiments: 18
Capacity valuation rows: 9
Freight experiments: 4
Demand experiments: 4
Sensitivity cost reconciliation: PASSED


,Experiment,Category,Change_Pct,Status,Optimized_Cost_USD,Cost_Change_USD
0,Reference,Reference,0,Optimal,404131.20,0.00
1,Supplier_SUP_A_Plus_10Pct,Supplier Capacity,10,Optimal,399345.20,-4786.00
2,Supplier_SUP_A_Plus_20Pct,Supplier Capacity,20,Optimal,396620.20,-7511.00
3,Supplier_SUP_A_Plus_30Pct,Supplier Capacity,30,Optimal,396248.20,-7883.00
4,Supplier_SUP_B_Plus_10Pct,Supplier Capacity,10,Optimal,404131.20,0.00
5,Supplier_SUP_B_Plus_20Pct,Supplier Capacity,20,Optimal,404131.20,0.00
6,Supplier_SUP_B_Plus_30Pct,Supplier Capacity,30,Optimal,404131.20,0.00
7,Manufacturing_Plus_5Pct,Manufacturing Capacity,5,Optimal,403891.20,-240.00
8,Manufacturing_Plus_10Pct,Manufacturing Capacity,10,Optimal,403775.20,-356.00
9,Manufacturing_Plus_20Pct,Manufacturing Capacity,20,Optimal,403623.20,-508.00



STEP 6B: PASSED


In [7]:
# STEP 7A: Export Tableau dashboard datasets

# Each entry defines:
# Dashboard folder, file, DataFrame, unique keys,
# business grain, and purpose.

tableau_exports = [

    # DASHBOARD 1: NETWORK COST
    (
        "01_cost",
        "network_kpis.csv",
        network_kpis,
        ["Plan"],
        "One row per plan",
        "Executive cost, savings and service KPIs"
    ),
    (
        "01_cost",
        "network_cost_categories.csv",
        network_cost_categories,
        ["Plan", "Cost_Category"],
        "One row per plan and cost category",
        "Baseline versus optimized network costs"
    ),
    (
        "01_cost",
        "network_savings_bridge.csv",
        network_savings_bridge,
        ["Cost_Category"],
        "One row per cost category",
        "Contribution of each category to total savings"
    ),

    # DASHBOARD 2: SOURCING AND MANUFACTURING
    (
        "02_sourcing",
        "supplier_weekly.csv",
        supplier_weekly,
        ["Supplier_ID", "Component_ID", "Order_Week"],
        "One row per supplier, component and order week",
        "Optimized supplier orders and capacity utilization"
    ),
    (
        "02_sourcing",
        "manufacturing_weekly.csv",
        manufacturing_weekly,
        ["Week"],
        "One row per production week",
        "Optimized production and plant capacity"
    ),
    (
        "02_sourcing",
        "component_inventory_weekly.csv",
        component_weekly,
        ["Component_ID", "Week"],
        "One row per component and week",
        "Component inventory and zero-buffer exposure"
    ),

    # DASHBOARD 3: TRANSPORTATION AND DISTRIBUTION
    (
        "03_distribution",
        "freight_modes.csv",
        freight_modes,
        ["Plan", "DC_ID", "Transport_Mode"],
        "One row per plan, DC and freight mode",
        "Standard versus expedited freight mix"
    ),
    (
        "03_distribution",
        "freight_lanes.csv",
        freight_lanes,
        ["Lane_ID"],
        "One row per transportation lane",
        "Baseline versus optimized lane decisions"
    ),
    (
        "03_distribution",
        "dc_weekly.csv",
        dc_weekly,
        ["Plan", "DC_ID", "Week"],
        "One row per plan, DC and week",
        "Customer service, shipping, inventory and storage"
    ),

    # DASHBOARD 4: SCENARIO AND SENSITIVITY
    (
        "04_risk",
        "scenario_summary.csv",
        scenario_summary,
        ["Scenario"],
        "One row per disruption scenario",
        "Disruption cost and operational impact"
    ),
    (
        "04_risk",
        "scenario_costs.csv",
        scenario_costs,
        ["Scenario", "Cost_Category"],
        "One row per scenario and cost category",
        "Scenario cost decomposition"
    ),
    (
        "04_risk",
        "sensitivity_summary.csv",
        sensitivity_summary,
        ["Experiment"],
        "One row per sensitivity experiment",
        "Experiment cost changes and solver status"
    ),
    (
        "04_risk",
        "sensitivity_costs.csv",
        sensitivity_costs,
        ["Experiment", "Cost_Category"],
        "One row per experiment and cost category",
        "Sensitivity cost decomposition"
    ),
    (
        "04_risk",
        "capacity_value.csv",
        capacity_value,
        ["Category", "Lever", "Change_Pct"],
        "One row per tested capacity expansion",
        "Capacity savings and investment break-even value"
    ),
    (
        "04_risk",
        "freight_sensitivity.csv",
        freight_sensitivity,
        ["Experiment"],
        "One row per freight experiment",
        "Freight price and operational decision effects"
    ),
    (
        "04_risk",
        "demand_sensitivity.csv",
        demand_sensitivity,
        ["Experiment"],
        "One row per demand-growth experiment",
        "Growth feasibility and incremental supply cost"
    )
]

assert len(tableau_exports) == 16

export_manifest = []

for (
    dashboard,
    filename,
    dataframe,
    keys,
    grain,
    description
) in tableau_exports:

    # Validate business grain
    check_grain(dataframe, keys, filename)

    # Export dataset
    path = dashboard_dirs[dashboard] / filename

    dataframe.to_csv(path, index=False)

    # Read exported file back for verification
    reloaded = pd.read_csv(path)

    assert len(reloaded) == len(dataframe)
    assert list(reloaded.columns) == list(dataframe.columns)

    export_manifest.append({
        "Dashboard": dashboard,
        "Filename": filename,
        "Relative_Path": str(
            path.relative_to(tableau_root)
        ),
        "Rows": len(dataframe),
        "Columns": len(dataframe.columns),
        "Unique_Keys": ", ".join(keys),
        "Grain": grain,
        "Description": description
    })

    print(
        f"EXPORTED: {dashboard}/{filename} "
        f"({len(dataframe)} rows)"
    )

manifest = pd.DataFrame(export_manifest)

assert len(manifest) == 16
assert manifest["Relative_Path"].is_unique

print("\n" + "=" * 50)
print("TABLEAU REPORTING EXPORT COMPLETE")
print("Files exported:", len(manifest))
print("Dashboard folders:", len(dashboard_dirs))
print("STEP 7A: PASSED")

display(manifest)


EXPORTED: 01_cost/network_kpis.csv (2 rows)
EXPORTED: 01_cost/network_cost_categories.csv (8 rows)
EXPORTED: 01_cost/network_savings_bridge.csv (4 rows)
EXPORTED: 02_sourcing/supplier_weekly.csv (72 rows)
EXPORTED: 02_sourcing/manufacturing_weekly.csv (12 rows)
EXPORTED: 02_sourcing/component_inventory_weekly.csv (36 rows)
EXPORTED: 03_distribution/freight_modes.csv (8 rows)
EXPORTED: 03_distribution/freight_lanes.csv (4 rows)
EXPORTED: 03_distribution/dc_weekly.csv (48 rows)
EXPORTED: 04_risk/scenario_summary.csv (5 rows)
EXPORTED: 04_risk/scenario_costs.csv (20 rows)
EXPORTED: 04_risk/sensitivity_summary.csv (18 rows)
EXPORTED: 04_risk/sensitivity_costs.csv (64 rows)
EXPORTED: 04_risk/capacity_value.csv (9 rows)
EXPORTED: 04_risk/freight_sensitivity.csv (4 rows)
EXPORTED: 04_risk/demand_sensitivity.csv (4 rows)

TABLEAU REPORTING EXPORT COMPLETE
Files exported: 16
Dashboard folders: 4
STEP 7A: PASSED


,Dashboard,Filename,Relative_Path,Rows,Columns,Unique_Keys,Grain,Description
0,01_cost,network_kpis.csv,01_cost/network_kpis.csv,2,16,Plan,One row per plan,"Executive cost, savings and service KPIs"
1,01_cost,network_cost_categories.csv,01_cost/network_cost_categories.csv,8,4,"Plan, Cost_Category",One row per plan and cost category,Baseline versus optimized network costs
2,01_cost,network_savings_bridge.csv,01_cost/network_savings_bridge.csv,4,6,Cost_Category,One row per cost category,Contribution of each category to total savings
3,02_sourcing,supplier_weekly.csv,02_sourcing/supplier_weekly.csv,72,14,"Supplier_ID, Component_ID, Order_Week","One row per supplier, component and order week",Optimized supplier orders and capacity utiliza...
4,02_sourcing,manufacturing_weekly.csv,02_sourcing/manufacturing_weekly.csv,12,8,Week,One row per production week,Optimized production and plant capacity
5,02_sourcing,component_inventory_weekly.csv,02_sourcing/component_inventory_weekly.csv,36,6,"Component_ID, Week",One row per component and week,Component inventory and zero-buffer exposure
6,03_distribution,freight_modes.csv,03_distribution/freight_modes.csv,8,6,"Plan, DC_ID, Transport_Mode","One row per plan, DC and freight mode",Standard versus expedited freight mix
7,03_distribution,freight_lanes.csv,03_distribution/freight_lanes.csv,4,11,Lane_ID,One row per transportation lane,Baseline versus optimized lane decisions
8,03_distribution,dc_weekly.csv,03_distribution/dc_weekly.csv,48,21,"Plan, DC_ID, Week","One row per plan, DC and week","Customer service, shipping, inventory and storage"
9,04_risk,scenario_summary.csv,04_risk/scenario_summary.csv,5,19,Scenario,One row per disruption scenario,Disruption cost and operational impact


In [8]:
# STEP 8A: Generate Tableau data dictionary

dictionary_lines = [
    "# Project 2 — Tableau Reporting Data Dictionary",
    "",
    "## Purpose",
    "Reporting datasets for the End-to-End Hardware "
    "Supply Network Optimizer portfolio project.",
    "",
    "All network quantities, costs and capacities "
    "are hypothetical.",
    "",
    "## Reporting Rules",
    "",
    "- Connect each CSV as an independent Tableau "
    "data source unless a relationship has been "
    "explicitly validated.",
    "- Do not sum executive KPIs across unrelated "
    "fact tables.",
    "- `Week` is a planning-period index from 1 to 12.",
    "- `Reference` means the original optimized model.",
    "- Scenario incremental costs are not expected "
    "losses or probability-weighted risk.",
    "- Sensitivity cost changes reflect reoptimization "
    "under changed assumptions.",
    "- Non-optimal experiments do not have verified "
    "optimized financial values.",
    "- Supplier and manufacturing tables represent "
    "the optimized reference solution.",
    "",
    "## Dataset Inventory",
    "",
    "| Dashboard | File | Rows | Grain | Purpose |",
    "|---|---|---:|---|---|"
]

for row in manifest.itertuples(index=False):

    dictionary_lines.append(
        f"| {row.Dashboard} "
        f"| `{row.Relative_Path}` "
        f"| {row.Rows} "
        f"| {row.Grain} "
        f"| {row.Description} |"
    )

dictionary_lines.extend([
    "",
    "## Recommended Tableau Relationships",
    "",
    "Use separate sources for the initial dashboards.",
    "Only create relationships when matching dimensions "
    "and source grains are understood.",
    "",
    "### Dashboard 1",
    "- Executive cards: `network_kpis.csv`",
    "- Cost comparison: `network_cost_categories.csv`",
    "- Savings waterfall: `network_savings_bridge.csv`",
    "",
    "### Dashboard 2",
    "- Supplier analysis: `supplier_weekly.csv`",
    "- Production utilization: `manufacturing_weekly.csv`",
    "- Inventory buffers: `component_inventory_weekly.csv`",
    "",
    "### Dashboard 3",
    "- Shipping mode mix: `freight_modes.csv`",
    "- Lane allocation: `freight_lanes.csv`",
    "- DC inventory and service: `dc_weekly.csv`",
    "",
    "### Dashboard 4",
    "- Disruption comparison: `scenario_summary.csv`",
    "- Disruption cost categories: `scenario_costs.csv`",
    "- Sensitivity experiments: `sensitivity_summary.csv`",
    "- Sensitivity cost categories: `sensitivity_costs.csv`",
    "- Capacity investment value: `capacity_value.csv`",
    "- Freight economics: `freight_sensitivity.csv`",
    "- Demand growth: `demand_sensitivity.csv`",
    "",
    "## Key Metric Definitions",
    "",
    "**Total network cost:** Procurement, manufacturing, "
    "freight and DC holding cost over 12 weeks.",
    "",
    "**Savings:** Baseline modeled cost minus "
    "optimized modeled cost.",
    "",
    "**Service level:** Fulfilled customer units "
    "divided by customer demand units.",
    "",
    "**Supplier utilization:** Ordered units "
    "divided by available weekly order capacity.",
    "",
    "**Storage utilization:** Inventory present "
    "before weekly customer fulfillment divided "
    "by DC storage capacity.",
    "",
    "**Capacity investment value:** Modeled network "
    "cost reduction before the cost of adding capacity.",
    "",
    "## Data Lineage",
    "",
    "- Notebook 03: Baseline planning",
    "- Notebook 04: Network optimization",
    "- Notebook 05: Sourcing and manufacturing",
    "- Notebook 06: Transportation and distribution",
    "- Notebook 07: Disruption scenarios",
    "- Notebook 08: Sensitivity and investment value",
    "- Notebook 09: Tableau reporting preparation",
    "",
    "## Important Limitations",
    "",
    "The model is deterministic. It does not establish "
    "actual future financial savings, capital costs, "
    "probabilistic disruption risk or real-world "
    "service performance."
])

data_dictionary = "\n".join(dictionary_lines)

dictionary_path = (
    tableau_root / "README.md"
)

dictionary_path.write_text(
    data_dictionary,
    encoding="utf-8"
)

assert dictionary_path.is_file()

print("DATA DICTIONARY CREATED")
print("File:", dictionary_path)
print("Documented datasets:", len(manifest))
print("STEP 8A: PASSED")


DATA DICTIONARY CREATED
File: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/outputs/tableau/README.md
Documented datasets: 16
STEP 8A: PASSED


In [9]:
# STEP 8B: Final Tableau dataset quality assurance

print("NOTEBOOK 09 — FINAL QUALITY ASSURANCE")
print("=" * 55)

# --------------------------------------------------
# 1. Financial consistency
# --------------------------------------------------

assert np.isclose(
    baseline_cost - optimized_cost,
    total_savings,
    atol=0.01
)

assert np.isclose(
    network_savings_bridge["Savings_USD"].sum(),
    total_savings,
    atol=0.01
)

print("Network financial reconciliation: PASSED")

# --------------------------------------------------
# 2. Reference model consistency
# --------------------------------------------------

reference_scenario_cost = scenario_summary.loc[
    scenario_summary["Scenario"] == "Reference",
    "Optimized_Cost_USD"
].iloc[0]

reference_sensitivity_cost = sensitivity_summary.loc[
    sensitivity_summary["Experiment"] == "Reference",
    "Optimized_Cost_USD"
].iloc[0]

assert np.isclose(
    reference_scenario_cost,
    optimized_cost,
    atol=0.01
)

assert np.isclose(
    reference_sensitivity_cost,
    optimized_cost,
    atol=0.01
)

print("Notebook 04/07/08 reconciliation: PASSED")

# --------------------------------------------------
# 3. Customer service and inventory
# --------------------------------------------------

assert dc_weekly["Service_Level_Pct"].eq(100).all()

assert (
    dc_weekly["Storage_Headroom_Units"] >= 0
).all()

print("Distribution service and storage: PASSED")

# --------------------------------------------------
# 4. Scenario and sensitivity audits
# --------------------------------------------------

assert (
    scenario_summary.loc[
        scenario_summary["Is_Optimal"] == 1,
        "Independent_Audit"
    ] == "PASSED"
).all()

assert (
    sensitivity_summary.loc[
        sensitivity_summary["Is_Optimal"] == 1,
        "Independent_Audit"
    ] == "PASSED"
).all()

print("Optimization scenario audits: PASSED")

# --------------------------------------------------
# 5. Export and reporting integrity
# --------------------------------------------------

assert len(manifest) == 16

for (
    dashboard, filename, dataframe, keys,
    grain, description
) in tableau_exports:

    path = dashboard_dirs[dashboard] / filename

    assert path.is_file()
    check_grain(dataframe, keys, filename)

    exported = pd.read_csv(path)

    assert len(exported) == len(dataframe)

print("All 16 CSV exports: PASSED")

# --------------------------------------------------
# 6. Generate Notebook 09 executive summary
# --------------------------------------------------

summary = f"""
# Notebook 09 — Tableau Reporting Summary

## Objective
Convert the validated analytical results of
Notebooks 01–08 into consistent Tableau-ready
reporting datasets.

## Reporting Architecture
- **4 dashboard subject areas**
- **16 Tableau-ready CSV datasets**
- **20 source tables loaded**
- **One data dictionary with dataset grains**
- **One reproducible reporting pipeline**

## Financial Reference
- Baseline cost: **${baseline_cost:,.2f}**
- Optimized cost: **${optimized_cost:,.2f}**
- Net modeled savings: **${total_savings:,.2f}**
- Modeled improvement:
  **{total_savings / baseline_cost * 100:.2f}%**

## Dashboard Coverage

### Dashboard 1 — Network Cost
Financial KPIs, cost-category comparisons,
and optimization savings decomposition.

### Dashboard 2 — Sourcing & Manufacturing
Optimized supplier ordering, manufacturing
utilization, and component inventory buffers.

### Dashboard 3 — Transportation & Distribution
Freight mode allocation, lane decisions,
warehouse inventory, storage and service.

### Dashboard 4 — Risk & Sensitivity
Disruption cost, operational response,
capacity investment value, freight economics,
and demand growth analysis.

## Validation
- Reference costs reconciled across models
- Financial categories reconciled to totals
- Distribution service and capacity validated
- Optimal scenario audits verified
- Export files reloaded successfully
- Dataset key uniqueness validated

## Deliverables
All datasets saved under:
`outputs/tableau/`

Data dictionary:
`outputs/tableau/README.md`

## Next Steps
1. Connect CSVs to Tableau Desktop.
2. Build four executive dashboards.
3. Add clear KPI definitions and business findings.
4. Export dashboard images for portfolio reporting.
5. Complete Notebook 10 and final GitHub documentation.
"""

display(Markdown(summary))

docs_dir = repo_root / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

summary_path = (
    docs_dir / "notebook09_findings.md"
)

summary_path.write_text(
    summary,
    encoding="utf-8"
)

assert summary_path.is_file()
assert dictionary_path.is_file()

print("\n" + "=" * 55)
print("NOTEBOOK 09: COMPLETE")
print("=" * 55)

print("Tableau datasets:", len(manifest))
print("Dashboard folders:", len(dashboard_dirs))
print("Data dictionary: README.md")
print("Executive summary: notebook09_findings.md")
print("All validation checks: PASSED")


NOTEBOOK 09 — FINAL QUALITY ASSURANCE
Network financial reconciliation: PASSED
Notebook 04/07/08 reconciliation: PASSED
Distribution service and storage: PASSED
Optimization scenario audits: PASSED
All 16 CSV exports: PASSED



# Notebook 09 — Tableau Reporting Summary

## Objective
Convert the validated analytical results of
Notebooks 01–08 into consistent Tableau-ready
reporting datasets.

## Reporting Architecture
- **4 dashboard subject areas**
- **16 Tableau-ready CSV datasets**
- **20 source tables loaded**
- **One data dictionary with dataset grains**
- **One reproducible reporting pipeline**

## Financial Reference
- Baseline cost: **$414,139.70**
- Optimized cost: **$404,131.20**
- Net modeled savings: **$10,008.50**
- Modeled improvement:
  **2.42%**

## Dashboard Coverage

### Dashboard 1 — Network Cost
Financial KPIs, cost-category comparisons,
and optimization savings decomposition.

### Dashboard 2 — Sourcing & Manufacturing
Optimized supplier ordering, manufacturing
utilization, and component inventory buffers.

### Dashboard 3 — Transportation & Distribution
Freight mode allocation, lane decisions,
warehouse inventory, storage and service.

### Dashboard 4 — Risk & Sensitivity
Disruption cost, operational response,
capacity investment value, freight economics,
and demand growth analysis.

## Validation
- Reference costs reconciled across models
- Financial categories reconciled to totals
- Distribution service and capacity validated
- Optimal scenario audits verified
- Export files reloaded successfully
- Dataset key uniqueness validated

## Deliverables
All datasets saved under:
`outputs/tableau/`

Data dictionary:
`outputs/tableau/README.md`

## Next Steps
1. Connect CSVs to Tableau Desktop.
2. Build four executive dashboards.
3. Add clear KPI definitions and business findings.
4. Export dashboard images for portfolio reporting.
5. Complete Notebook 10 and final GitHub documentation.



NOTEBOOK 09: COMPLETE
Tableau datasets: 16
Dashboard folders: 4
Data dictionary: README.md
Executive summary: notebook09_findings.md
All validation checks: PASSED
